# Pretraining — Expert
> Section 01 — Topic 05 — expert

**Prerequisites:** [05-pretraining/foundations.ipynb](./foundations.ipynb), [05-pretraining/advanced.ipynb](./advanced.ipynb)

**What you'll learn:**
- Scaling laws: Kaplan et al. and Chinchilla, and how to use them
- Compute-optimal training: sizing your model for your GPU budget
- Data mixtures and weighted domain sampling
- Data quality: MinHash deduplication and perplexity-based filtering
- Training dynamics: loss spikes, gradient norm monitoring
- Emergent abilities, phase transitions, and the controversy around them

**What you'll build:**
- Scaling law predictors, data quality filters, training monitors

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math
import hashlib
import struct
from typing import List, Set, Tuple, Dict
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 1. Scaling Laws

Scaling laws describe how language model performance (measured by cross-entropy loss) improves as you increase model size, dataset size, and compute budget. Understanding these laws is essential for making informed decisions about how to allocate resources for pretraining.

**Kaplan et al. (2020)** from OpenAI established that loss follows a power-law relationship with model parameters (N), dataset size (D), and compute (C). Specifically: L(N) ~ N^(-0.076), L(D) ~ D^(-0.095), and L(C) ~ C^(-0.050). A key finding was that model size matters more than dataset size — for a fixed compute budget, it is better to train a larger model on less data. This led to training very large models (like GPT-3 at 175B parameters) on relatively modest datasets (300B tokens).

**Chinchilla (Hoffmann et al., 2022)** from DeepMind challenged this conclusion. By training over 400 models of different sizes, they found that Kaplan's analysis was flawed because it did not train smaller models to convergence. The Chinchilla-optimal scaling suggests that model size and dataset size should scale equally — for every doubling of parameters, you should also double the training tokens. This means a 70B model should be trained on ~1.4T tokens. Chinchilla (70B params, 1.4T tokens) outperformed the much larger Gopher (280B params, 300B tokens) at one-quarter the inference cost.

The practical impact was enormous: the industry shifted from training massive undertrained models to training smaller, better-trained models. LLaMA (65B, 1.4T tokens), Mistral (7B, unreported but large), and most modern open-source models follow Chinchilla scaling or even overtrain relative to it.

In [ ]:
# Kaplan et al. scaling law formulas
def kaplan_loss_N(N: float, N_c: float = 8.8e13, alpha_N: float = 0.076) -> float:
    """Loss as a function of model parameters (Kaplan et al.)."""
    return (N_c / N) ** alpha_N

def kaplan_loss_D(D: float, D_c: float = 5.4e13, alpha_D: float = 0.095) -> float:
    """Loss as a function of dataset size in tokens (Kaplan et al.)."""
    return (D_c / D) ** alpha_D

def kaplan_loss_C(C: float, C_c: float = 3.1e8, alpha_C: float = 0.050) -> float:
    """Loss as a function of compute in PF-days (Kaplan et al.)."""
    return (C_c / C) ** alpha_C

# Chinchilla scaling law
def chinchilla_loss(N: float, D: float, E: float = 1.69, A: float = 406.4, B: float = 410.7, alpha: float = 0.34, beta: float = 0.28) -> float:
    """
    Chinchilla parametric loss: L(N, D) = E + A/N^alpha + B/D^beta
    N: parameters, D: tokens
    """
    return E + A / (N ** alpha) + B / (D ** beta)

def chinchilla_optimal_tokens(N: float, ratio: float = 20.0) -> float:
    """Chinchilla-optimal token count: D = ratio * N (default ratio ~20)."""
    return ratio * N

print("Scaling law functions defined.")
print(f"\nChinchilla loss for 7B model, 1.4T tokens: {chinchilla_loss(7e9, 1.4e12):.3f}")
print(f"Chinchilla loss for 70B model, 1.4T tokens: {chinchilla_loss(70e9, 1.4e12):.3f}")
print(f"Chinchilla loss for 70B model, 300B tokens: {chinchilla_loss(70e9, 300e9):.3f}")

In [ ]:
# Visualize scaling laws
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss vs model size
params = np.logspace(6, 12, 200)  # 1M to 1T parameters
losses_N = [kaplan_loss_N(n) for n in params]
axes[0].loglog(params, losses_N, color='steelblue', linewidth=2)
# Mark notable models
models = {'GPT-2': 1.5e9, 'GPT-3': 175e9, 'LLaMA-7B': 7e9, 'LLaMA-70B': 70e9}
for name, n in models.items():
    axes[0].scatter([n], [kaplan_loss_N(n)], s=60, zorder=5)
    axes[0].annotate(name, (n, kaplan_loss_N(n)), textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[0].set_xlabel('Parameters (N)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Kaplan: Loss vs Model Size')
axes[0].grid(True, alpha=0.3)

# Loss vs dataset size
tokens = np.logspace(8, 14, 200)
losses_D = [kaplan_loss_D(d) for d in tokens]
axes[1].loglog(tokens, losses_D, color='darkorange', linewidth=2)
axes[1].set_xlabel('Tokens (D)')
axes[1].set_ylabel('Loss')
axes[1].set_title('Kaplan: Loss vs Dataset Size')
axes[1].grid(True, alpha=0.3)

# Chinchilla: optimal allocation
# For a fixed compute budget, show how Kaplan vs Chinchilla allocate
model_sizes = np.logspace(8, 11, 100)
for budget_label, compute_tokens in [('100B tokens budget', 100e9), ('1T tokens budget', 1e12), ('10T tokens budget', 10e12)]:
    losses = [chinchilla_loss(n, compute_tokens / (6 * n) * 6 * n / n) for n in model_sizes]
    # Simplify: for fixed C = 6ND, D = C/(6N)
    C = 6 * 7e9 * float(compute_tokens.real if hasattr(compute_tokens, 'real') else compute_tokens)
    losses = [chinchilla_loss(n, C / (6 * n)) for n in model_sizes]
    axes[2].semilogx(model_sizes, losses, label=budget_label, linewidth=2)

axes[2].set_xlabel('Parameters (N)')
axes[2].set_ylabel('Chinchilla Loss')
axes[2].set_title('Chinchilla: Optimal Model Size per Budget')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Compute-Optimal Training

Given a fixed GPU budget (measured in GPU-hours or FLOPs), how should you allocate compute between model size and training tokens? This is the central question of compute-optimal training.

The compute required for training is approximately C = 6 * N * D, where N is model parameters and D is training tokens. The factor of 6 accounts for the forward pass (2ND FLOPs), backward pass (4ND FLOPs, approximately 2x forward). Given a compute budget C, you must choose N and D such that 6ND = C and the resulting loss L(N, D) is minimized.

Under Chinchilla scaling, the optimal ratio is approximately D = 20N. This means a 1B model should train on 20B tokens, a 7B model on 140B tokens, and a 70B model on 1.4T tokens. In practice, many teams "overtrain" relative to Chinchilla — training on more tokens than optimal — because inference cost is proportional to N but training is a one-time cost. LLaMA-2-7B was trained on 2T tokens (nearly 300x parameters), far beyond Chinchilla-optimal, because Meta wanted the smallest possible model with the best possible quality for cheap inference.

In [ ]:
def compute_budget_to_flops(gpu_hours: float, gpu_tflops: float = 312.0) -> float:
    """Convert GPU-hours to FLOPs. Default: A100 at 312 TFLOPS bf16."""
    return gpu_hours * 3600 * gpu_tflops * 1e12

def chinchilla_optimal_allocation(C_flops: float, ratio: float = 20.0) -> Tuple[float, float]:
    """
    Given compute budget in FLOPs, find optimal (N, D).
    C = 6 * N * D, D = ratio * N
    => C = 6 * N * ratio * N = 6 * ratio * N^2
    => N = sqrt(C / (6 * ratio))
    """
    N_opt = math.sqrt(C_flops / (6 * ratio))
    D_opt = ratio * N_opt
    return N_opt, D_opt

def estimate_training_time(N: float, D: float, num_gpus: int, gpu_tflops: float = 312.0, mfu: float = 0.4) -> float:
    """Estimate training time in hours. MFU = model FLOP utilization."""
    total_flops = 6 * N * D
    effective_tflops = num_gpus * gpu_tflops * mfu
    seconds = total_flops / (effective_tflops * 1e12)
    return seconds / 3600

# Example: 1000 A100 GPU-hours budget
budgets = [100, 1000, 10000, 100000]  # GPU-hours on A100
print(f"{'GPU-hours':>12} | {'Optimal N':>14} | {'Optimal D':>14} | {'N (human)':>12} | {'D (human)':>12}")
print('-' * 80)
for hours in budgets:
    C = compute_budget_to_flops(hours)
    N, D = chinchilla_optimal_allocation(C)
    N_str = f"{N/1e6:.0f}M" if N < 1e9 else f"{N/1e9:.1f}B"
    D_str = f"{D/1e9:.0f}B" if D < 1e12 else f"{D/1e12:.1f}T"
    print(f"{hours:>12,} | {N:>14.2e} | {D:>14.2e} | {N_str:>12} | {D_str:>12}")

In [ ]:
# Visualize: for a fixed budget, plot loss vs model size to find the optimum
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

C_flops = compute_budget_to_flops(10000)  # 10K A100-hours
model_sizes = np.logspace(7, 11, 500)
losses = []
for n in model_sizes:
    d = C_flops / (6 * n)  # D = C / (6N)
    if d > 0:
        losses.append(chinchilla_loss(n, d))
    else:
        losses.append(float('inf'))

losses = np.array(losses)
opt_idx = np.argmin(losses)
opt_N = model_sizes[opt_idx]
opt_D = C_flops / (6 * opt_N)

axes[0].semilogx(model_sizes, losses, color='steelblue', linewidth=2)
axes[0].scatter([opt_N], [losses[opt_idx]], color='red', s=100, zorder=5, label=f'Optimal: {opt_N/1e9:.1f}B params')
axes[0].set_xlabel('Model Parameters (N)')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'Optimal Model Size for 10K A100-hours')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot the token/param ratio of published models
published_models = {
    'GPT-3': (175e9, 300e9),
    'Chinchilla': (70e9, 1.4e12),
    'LLaMA-7B': (7e9, 1e12),
    'LLaMA-65B': (65e9, 1.4e12),
    'LLaMA-2-7B': (7e9, 2e12),
    'Mistral-7B': (7e9, 8e12),  # estimated
}

names = list(published_models.keys())
ratios = [d / n for n, d in published_models.values()]
colors = ['red' if r < 20 else 'steelblue' if r < 50 else 'forestgreen' for r in ratios]

axes[1].barh(names, ratios, color=colors, edgecolor='white')
axes[1].axvline(20, color='black', linestyle='--', label='Chinchilla optimal (20x)', linewidth=2)
axes[1].set_xlabel('Tokens / Parameters Ratio')
axes[1].set_title('Training Token Ratio: Published Models')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\nOptimal allocation: {opt_N/1e9:.1f}B params, {opt_D/1e12:.2f}T tokens")

## 3. Data Mixtures

The pretraining corpus is not a homogeneous blob of text. It is a mixture of domains — web crawls, books, scientific papers, code, Wikipedia, social media, and more — each with different quality levels, vocabulary distributions, and information density. The data mixture — how much weight to give each domain — significantly impacts model quality and capabilities.

The simplest approach is proportional sampling: draw from each domain in proportion to its size. But this is rarely optimal. Web crawl data is vast but noisy; books are small but high-quality; code is critical for reasoning but represents a small fraction of text on the internet. Most frontier labs use carefully tuned sampling weights that oversample high-quality domains relative to their natural proportion.

LLaMA-1 used approximately: 67% CommonCrawl, 15% C4, 4.5% Github, 4.5% Wikipedia, 4.5% books, 2.5% ArXiv, 2% StackExchange. LLaMA-2 increased the code and conversational data proportions. The optimal mixture depends on the intended use case — a model for code generation needs more code, a model for medical applications needs more biomedical text.

Implementing weighted sampling requires a sampler that draws from each domain with the specified probability, regardless of the domain's actual size. This can be done by maintaining a separate dataloader per domain and randomly choosing which domain to sample from at each step, or by using a weighted random sampler.

In [ ]:
class DomainMixtureSampler:
    """
    Samples batches from multiple domains according to specified weights.
    Each domain is a list of token sequences.
    """
    def __init__(self, domains: Dict[str, List[List[int]]], weights: Dict[str, float]):
        self.domain_names = list(domains.keys())
        self.domains = domains
        # Normalize weights
        total = sum(weights[d] for d in self.domain_names)
        self.probs = np.array([weights[d] / total for d in self.domain_names])
        # Track position within each domain
        self.positions = {d: 0 for d in self.domain_names}

    def sample_batch(self, batch_size: int) -> Tuple[List[List[int]], List[str]]:
        """Sample a batch, returning (sequences, domain_labels)."""
        # Choose domains for each item in the batch
        chosen_domains = np.random.choice(self.domain_names, size=batch_size, p=self.probs)
        sequences = []
        labels = []
        for domain_name in chosen_domains:
            data = self.domains[domain_name]
            idx = self.positions[domain_name] % len(data)
            sequences.append(data[idx])
            labels.append(domain_name)
            self.positions[domain_name] += 1
        return sequences, labels

    def get_stats(self, num_samples: int = 10000) -> Dict[str, float]:
        """Sample many times and report actual domain distribution."""
        _, labels = self.sample_batch(num_samples)
        counts = defaultdict(int)
        for l in labels:
            counts[l] += 1
        return {d: counts[d] / num_samples for d in self.domain_names}

# Create synthetic domain data
np.random.seed(42)
domains = {
    'web': [np.random.randint(0, 5000, 128).tolist() for _ in range(10000)],
    'books': [np.random.randint(0, 5000, 128).tolist() for _ in range(500)],
    'code': [np.random.randint(0, 5000, 128).tolist() for _ in range(2000)],
    'wikipedia': [np.random.randint(0, 5000, 128).tolist() for _ in range(800)],
    'arxiv': [np.random.randint(0, 5000, 128).tolist() for _ in range(300)],
}

# LLaMA-style weights (simplified)
weights = {'web': 0.67, 'books': 0.05, 'code': 0.10, 'wikipedia': 0.10, 'arxiv': 0.08}

sampler = DomainMixtureSampler(domains, weights)
stats = sampler.get_stats(50000)

print("Domain sampling distribution:")
print(f"{'Domain':<12} | {'Target':>8} | {'Actual':>8} | {'Natural (by size)':>18}")
print('-' * 55)
total_docs = sum(len(d) for d in domains.values())
for d in sampler.domain_names:
    natural = len(domains[d]) / total_docs
    print(f"{d:<12} | {weights[d]:>7.1%} | {stats[d]:>7.1%} | {natural:>17.1%}")

In [ ]:
# Visualize domain mixture: natural vs weighted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

domain_names = list(domains.keys())
natural = [len(domains[d]) / total_docs for d in domain_names]
weighted = [weights[d] for d in domain_names]

x = np.arange(len(domain_names))
width = 0.35

axes[0].bar(x - width/2, natural, width, label='Natural (by size)', color='lightcoral')
axes[0].bar(x + width/2, weighted, width, label='Weighted (curated)', color='steelblue')
axes[0].set_xticks(x)
axes[0].set_xticklabels(domain_names)
axes[0].set_ylabel('Sampling Proportion')
axes[0].set_title('Natural vs Curated Domain Weights')
axes[0].legend()

# Show upsampling factors
upsample_factors = [w / n if n > 0 else 0 for w, n in zip(weighted, natural)]
colors = ['forestgreen' if f > 1 else 'lightcoral' for f in upsample_factors]
axes[1].bar(domain_names, upsample_factors, color=colors, edgecolor='white')
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1, label='No change')
axes[1].set_ylabel('Upsampling Factor')
axes[1].set_title('Upsampling Factor per Domain')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Data Quality — Deduplication and Filtering

Data quality has proven to be at least as important as data quantity for pretraining. Two critical data quality techniques are **deduplication** (removing near-duplicate documents) and **perplexity-based filtering** (removing low-quality documents using a reference model).

**MinHash deduplication** efficiently identifies near-duplicate documents in massive corpora. The algorithm works by: (1) representing each document as a set of n-grams (character or word level), (2) computing multiple hash functions over these n-gram sets to create a "MinHash signature" — a compact fingerprint where the probability that two documents share a signature value equals their Jaccard similarity, (3) using Locality-Sensitive Hashing (LSH) to group documents with similar signatures into buckets, and (4) removing all but one document from each group of near-duplicates. This is O(n) rather than the O(n^2) required for all-pairs comparison. The Pile, RedPajama, and DCLM all use MinHash deduplication as a core step.

**Perplexity-based filtering** uses a reference language model (typically a small, well-trained model like a KenLM n-gram model trained on Wikipedia) to score each document. Documents with very high perplexity are likely gibberish, spam, or machine-generated text. Documents with very low perplexity may be boilerplate or overly repetitive. The sweet spot is medium perplexity — text that is natural and informative but not trivially predictable. CCNet (the pipeline behind CC100 and CulturaX) uses this approach, filtering by both perplexity and language identification.

In [ ]:
class MinHashDeduplicator:
    """
    MinHash-based near-duplicate detection.
    Implements the full pipeline: shingling -> MinHash -> LSH banding.
    """
    def __init__(self, num_hashes: int = 128, num_bands: int = 16, ngram_size: int = 5):
        self.num_hashes = num_hashes
        self.num_bands = num_bands
        self.rows_per_band = num_hashes // num_bands
        self.ngram_size = ngram_size
        # Random hash parameters: h(x) = (a*x + b) % p
        self.max_hash = 2**32 - 1
        self.prime = 4294967311  # Large prime > max_hash
        np.random.seed(42)
        self.a = np.random.randint(1, self.prime, size=num_hashes).astype(np.uint64)
        self.b = np.random.randint(0, self.prime, size=num_hashes).astype(np.uint64)

    def _get_shingles(self, text: str) -> Set[int]:
        """Convert text to a set of hashed character n-grams."""
        shingles = set()
        for i in range(len(text) - self.ngram_size + 1):
            shingle = text[i:i + self.ngram_size]
            h = int(hashlib.md5(shingle.encode()).hexdigest()[:8], 16)
            shingles.add(h)
        return shingles

    def compute_minhash(self, text: str) -> np.ndarray:
        """Compute MinHash signature for a document."""
        shingles = self._get_shingles(text)
        if not shingles:
            return np.full(self.num_hashes, self.max_hash, dtype=np.uint64)

        shingle_array = np.array(list(shingles), dtype=np.uint64)
        # For each hash function, compute min hash over all shingles
        signature = np.full(self.num_hashes, self.max_hash, dtype=np.uint64)
        for s in shingle_array:
            hashes = (self.a * s + self.b) % self.prime
            signature = np.minimum(signature, hashes)
        return signature

    def jaccard_from_minhash(self, sig1: np.ndarray, sig2: np.ndarray) -> float:
        """Estimate Jaccard similarity from MinHash signatures."""
        return np.mean(sig1 == sig2)

    def find_duplicates(self, documents: List[str], threshold: float = 0.5) -> List[Set[int]]:
        """
        Find groups of near-duplicate documents using LSH banding.
        Returns list of sets, each set containing indices of duplicates.
        """
        # Compute all signatures
        signatures = [self.compute_minhash(doc) for doc in documents]

        # LSH: group by band hashes
        candidate_pairs = set()
        for band_idx in range(self.num_bands):
            start = band_idx * self.rows_per_band
            end = start + self.rows_per_band
            buckets = defaultdict(list)
            for doc_idx, sig in enumerate(signatures):
                band_hash = hash(tuple(sig[start:end]))
                buckets[band_hash].append(doc_idx)
            for bucket in buckets.values():
                if len(bucket) > 1:
                    for i in range(len(bucket)):
                        for j in range(i + 1, len(bucket)):
                            candidate_pairs.add((bucket[i], bucket[j]))

        # Verify candidates with actual Jaccard estimate
        from collections import defaultdict as dd
        graph = dd(set)
        for i, j in candidate_pairs:
            sim = self.jaccard_from_minhash(signatures[i], signatures[j])
            if sim >= threshold:
                graph[i].add(j)
                graph[j].add(i)

        # Connected components (duplicate groups)
        visited = set()
        groups = []
        for node in graph:
            if node not in visited:
                group = set()
                stack = [node]
                while stack:
                    n = stack.pop()
                    if n not in visited:
                        visited.add(n)
                        group.add(n)
                        stack.extend(graph[n] - visited)
                groups.append(group)

        return groups

# Test with synthetic documents
docs = [
    "The quick brown fox jumps over the lazy dog near the river bank.",
    "The quick brown fox jumps over the lazy dog near the river bank!",  # near-duplicate
    "The quick brown fox leaps over the lazy dog near the river bank.",  # near-duplicate
    "Machine learning is a subset of artificial intelligence focused on data.",
    "Machine learning is a subset of artificial intelligence focusing on data.",  # near-duplicate
    "Quantum computing leverages quantum mechanical phenomena for computation.",
    "The weather today is sunny with a high of 75 degrees Fahrenheit.",
]

dedup = MinHashDeduplicator(num_hashes=128, num_bands=16, ngram_size=5)
groups = dedup.find_duplicates(docs, threshold=0.3)

print("Near-duplicate groups found:")
for i, group in enumerate(groups):
    print(f"\nGroup {i+1}:")
    for idx in group:
        print(f"  [{idx}] {docs[idx][:80]}...")

In [ ]:
class PerplexityFilter:
    """
    Perplexity-based document filtering using a reference LM.
    In production, this would use KenLM. Here we use a simple n-gram model.
    """
    def __init__(self, n: int = 3):
        self.n = n
        self.ngram_counts = defaultdict(int)
        self.context_counts = defaultdict(int)
        self.vocab_size = 0

    def fit(self, texts: List[str]):
        """Train the n-gram model on reference texts."""
        all_chars = set()
        for text in texts:
            all_chars.update(text)
            for i in range(len(text) - self.n + 1):
                ngram = text[i:i + self.n]
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
        self.vocab_size = len(all_chars)

    def perplexity(self, text: str) -> float:
        """Compute perplexity of text under the reference model."""
        if len(text) < self.n:
            return float('inf')
        log_prob = 0.0
        count = 0
        for i in range(len(text) - self.n + 1):
            ngram = text[i:i + self.n]
            context = ngram[:-1]
            # Laplace smoothing
            prob = (self.ngram_counts[ngram] + 1) / (self.context_counts[context] + self.vocab_size)
            log_prob += math.log(prob)
            count += 1
        avg_log_prob = log_prob / max(count, 1)
        return math.exp(-avg_log_prob)

    def filter_documents(
        self, documents: List[str], low_pct: float = 10, high_pct: float = 90
    ) -> Tuple[List[str], List[str], List[float]]:
        """Filter documents, keeping those within perplexity percentile range."""
        perplexities = [self.perplexity(doc) for doc in documents]
        low_thresh = np.percentile(perplexities, low_pct)
        high_thresh = np.percentile(perplexities, high_pct)

        kept = []
        removed = []
        for doc, ppl in zip(documents, perplexities):
            if low_thresh <= ppl <= high_thresh:
                kept.append(doc)
            else:
                removed.append(doc)

        return kept, removed, perplexities

# Create reference corpus ("good" text) and test corpus
reference = [
    "Language models predict the next word in a sequence of text.",
    "Neural networks learn representations from data through backpropagation.",
    "Transformers use self-attention mechanisms for sequence modeling.",
    "The training process optimizes model parameters to minimize loss.",
    "Deep learning has revolutionized natural language processing.",
] * 20  # Repeat to get decent statistics

test_docs = [
    "Transformers have become the dominant architecture for language modeling.",
    "aaaa bbbb cccc dddd eeee ffff gggg hhhh iiii jjjj",  # gibberish
    "The attention mechanism computes weighted sums of value vectors.",
    "buy cheap viagra online best price guaranteed click here now!!!",  # spam
    "Gradient descent iteratively updates parameters toward lower loss.",
    "the the the the the the the the the the the the the",  # repetitive
    "Modern LLMs are trained on trillions of tokens from diverse sources.",
    "Copyright 2024 All Rights Reserved Terms and Conditions Apply",  # boilerplate
]

ppl_filter = PerplexityFilter(n=4)
ppl_filter.fit(reference)

kept, removed, perplexities = ppl_filter.filter_documents(test_docs)

print("Perplexity scores:")
for doc, ppl in sorted(zip(test_docs, perplexities), key=lambda x: x[1]):
    status = 'KEEP' if doc in kept else 'REMOVE'
    print(f"  [{status:>6}] PPL={ppl:>8.1f} | {doc[:65]}")

## 5. Training Dynamics

Understanding training dynamics — how loss, gradients, and other metrics evolve during pretraining — is essential for monitoring training runs and catching problems early. Production pretraining teams monitor dozens of metrics in real-time dashboards.

**Loss spikes** are the most feared training event. They occur when a batch triggers an unusually large gradient update, often caused by rare or corrupted data. Minor spikes (loss increases by 10-20% and recovers within 100 steps) are normal and harmless. Major spikes (loss doubles or more) can permanently damage model quality by pushing parameters into a bad region. The standard mitigations are gradient clipping (capping the gradient norm, typically at 1.0), z-loss regularization (penalizing large logits), and data filtering (removing problematic examples).

**Gradient norms** should be monitored continuously. A healthy training run shows gradient norms that are initially large (during warmup), stabilize during the main training phase, and may increase slightly toward the end. A sudden spike in gradient norm often precedes a loss spike. If gradient norms grow steadily, the model may be headed toward instability. The ratio of gradient norm to parameter norm (the "update ratio") should typically be around 1e-3 to 1e-2.

**Layer-wise statistics** can reveal problems that global metrics miss. If one layer has much larger gradients or activations than others, it may be a bottleneck or source of instability. Monitoring the mean and standard deviation of activations per layer helps catch issues like vanishing gradients (early layers have near-zero activations) or exploding activations (later layers have very large values).

In [ ]:
class TrainingMonitor:
    """
    Monitors training dynamics: loss, gradient norms, activation stats.
    """
    def __init__(self):
        self.history = {
            'loss': [],
            'grad_norm': [],
            'param_norm': [],
            'update_ratio': [],
            'layer_grad_norms': defaultdict(list),
            'loss_spikes': [],
        }
        self.ema_loss = None
        self.ema_alpha = 0.01
        self.spike_threshold = 2.0  # detect spikes > 2x EMA

    def log_step(self, model: nn.Module, loss: float, step: int):
        """Log metrics for one training step."""
        # Loss
        self.history['loss'].append(loss)

        # Detect loss spikes
        if self.ema_loss is None:
            self.ema_loss = loss
        else:
            if loss > self.spike_threshold * self.ema_loss:
                self.history['loss_spikes'].append({
                    'step': step, 'loss': loss, 'ema': self.ema_loss,
                    'ratio': loss / self.ema_loss
                })
            self.ema_loss = self.ema_alpha * loss + (1 - self.ema_alpha) * self.ema_loss

        # Gradient norm (total and per-layer)
        total_grad_norm = 0.0
        total_param_norm = 0.0
        for name, param in model.named_parameters():
            if param.grad is not None:
                grad_norm = param.grad.data.norm(2).item()
                param_norm = param.data.norm(2).item()
                total_grad_norm += grad_norm ** 2
                total_param_norm += param_norm ** 2
                # Track per-layer norms
                layer_name = name.split('.')[0]
                self.history['layer_grad_norms'][name].append(grad_norm)

        total_grad_norm = math.sqrt(total_grad_norm)
        total_param_norm = math.sqrt(total_param_norm)
        update_ratio = total_grad_norm / (total_param_norm + 1e-8)

        self.history['grad_norm'].append(total_grad_norm)
        self.history['param_norm'].append(total_param_norm)
        self.history['update_ratio'].append(update_ratio)

    def report(self):
        """Print summary of training dynamics."""
        print(f"Training steps: {len(self.history['loss'])}")
        print(f"Final loss: {self.history['loss'][-1]:.4f}")
        print(f"Loss spikes detected: {len(self.history['loss_spikes'])}")
        for spike in self.history['loss_spikes']:
            print(f"  Step {spike['step']}: loss={spike['loss']:.4f} ({spike['ratio']:.1f}x EMA)")
        print(f"Avg gradient norm: {np.mean(self.history['grad_norm']):.4f}")
        print(f"Avg update ratio: {np.mean(self.history['update_ratio']):.6f}")

print("TrainingMonitor defined.")

In [ ]:
# Train a small model with the monitor
model = MiniTransformerLM(vocab_size=5000, d_model=128, n_heads=4, n_layers=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
monitor = TrainingMonitor()

model.train()
for step in range(200):
    batch = torch.randint(0, 5000, (8, 64), device=device)
    logits = model(batch)

    # Shift for CLM loss
    loss = F.cross_entropy(
        logits[:, :-1].contiguous().view(-1, 5000),
        batch[:, 1:].contiguous().view(-1)
    )

    # Inject artificial spike at step 80
    if step == 80:
        loss = loss * 5.0

    loss.backward()
    monitor.log_step(model, loss.item(), step)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()

monitor.report()

In [ ]:
# Visualize training dynamics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

steps = range(len(monitor.history['loss']))

# Loss with spike markers
axes[0, 0].plot(steps, monitor.history['loss'], color='steelblue', alpha=0.7, linewidth=0.8)
for spike in monitor.history['loss_spikes']:
    axes[0, 0].axvline(spike['step'], color='red', alpha=0.5, linestyle='--')
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss (red = detected spikes)')

# Gradient norm
axes[0, 1].plot(steps, monitor.history['grad_norm'], color='darkorange', alpha=0.7, linewidth=0.8)
axes[0, 1].set_xlabel('Step')
axes[0, 1].set_ylabel('Gradient Norm')
axes[0, 1].set_title('Gradient Norm Over Training')

# Update ratio
axes[1, 0].semilogy(steps, monitor.history['update_ratio'], color='forestgreen', alpha=0.7, linewidth=0.8)
axes[1, 0].axhline(1e-3, color='black', linestyle='--', alpha=0.5, label='Typical range')
axes[1, 0].axhline(1e-2, color='black', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Step')
axes[1, 0].set_ylabel('Update Ratio (grad/param norm)')
axes[1, 0].set_title('Update Ratio')
axes[1, 0].legend()

# Per-layer gradient norms (last 50 steps)
layer_names = list(monitor.history['layer_grad_norms'].keys())[:6]
for name in layer_names:
    norms = monitor.history['layer_grad_norms'][name][-50:]
    axes[1, 1].plot(norms, label=name.split('.')[-1][:15], alpha=0.7)
axes[1, 1].set_xlabel('Step (last 50)')
axes[1, 1].set_ylabel('Gradient Norm')
axes[1, 1].set_title('Per-Layer Gradient Norms')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Emergent Abilities and Phase Transitions

One of the most debated phenomena in LLM scaling is **emergent abilities** — capabilities that appear to suddenly materialize as model size crosses a threshold, seemingly absent in smaller models. The term was popularized by Wei et al. (2022), who documented cases where performance on certain benchmarks (like multi-step arithmetic, word unscrambling, and chain-of-thought reasoning) jumped from near-random to high accuracy over a narrow range of model sizes.

The original claim was provocative: it suggested that larger models are not just quantitatively better but qualitatively different, gaining entirely new capabilities through scale alone. This has profound implications for AI safety (capabilities might appear unpredictably) and for scaling decisions (you cannot extrapolate from small models to predict large model capabilities).

**The controversy:** Schaeffer et al. (2023) challenged this narrative in a highly influential paper, arguing that emergent abilities are a "mirage" created by the choice of evaluation metric. When benchmarks use discontinuous metrics (like exact-match accuracy, which is 0 until the model gets the entire answer right), performance appears to jump suddenly. When the same tasks are evaluated with continuous metrics (like token-level accuracy or edit distance), performance improves smoothly and predictably with scale. The authors showed that for every claimed emergent ability, a change of metric eliminated the apparent emergence.

The truth is likely nuanced. The metric-choice explanation is compelling for many benchmarks, but there are genuine phase transitions in learning dynamics — for instance, the sudden transition from memorization to generalization ("grokking"), or the point at which in-context learning first becomes reliable. Whether these constitute "emergence" in a meaningful sense remains an active debate.

In [ ]:
# Simulate the "emergence is a mirage" argument
# Show how the same underlying improvement looks different under different metrics

def sigmoid(x, center, steepness):
    return 1 / (1 + np.exp(-steepness * (x - center)))

# Underlying per-token accuracy (improves smoothly with scale)
model_sizes = np.logspace(7, 11, 100)  # 10M to 100B
per_token_acc = 0.3 + 0.6 * (np.log10(model_sizes) - 7) / 4  # Linear in log-scale
per_token_acc = np.clip(per_token_acc, 0, 0.95)

# Exact-match accuracy: requires ALL tokens correct
# If answer has k tokens each with per_token_acc probability:
answer_lengths = [5, 10, 20]  # Different task complexities
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, k in enumerate(answer_lengths):
    exact_match = per_token_acc ** k

    axes[ax_idx].semilogx(model_sizes, per_token_acc, '--', color='steelblue',
                          linewidth=2, label='Per-token accuracy (continuous)')
    axes[ax_idx].semilogx(model_sizes, exact_match, '-', color='red',
                          linewidth=2, label=f'Exact match ({k} tokens)')
    axes[ax_idx].set_xlabel('Model Parameters')
    axes[ax_idx].set_ylabel('Accuracy')
    axes[ax_idx].set_title(f'Answer Length = {k} tokens')
    axes[ax_idx].legend(fontsize=9)
    axes[ax_idx].set_ylim(-0.05, 1.05)
    axes[ax_idx].grid(True, alpha=0.3)

plt.suptitle('"Emergent" Abilities: Continuous vs Discontinuous Metrics (Schaeffer et al., 2023)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize published results on emergent abilities
# Data approximated from Wei et al. (2022) Figure 2

tasks = {
    'Multi-step Arithmetic': {
        'sizes': [0.4e9, 1.3e9, 7.1e9, 13e9, 70e9, 175e9, 540e9],
        'accuracy': [0.0, 0.0, 0.02, 0.05, 0.15, 0.45, 0.85]
    },
    'Word Unscrambling': {
        'sizes': [0.4e9, 1.3e9, 7.1e9, 13e9, 70e9, 175e9, 540e9],
        'accuracy': [0.0, 0.01, 0.02, 0.05, 0.35, 0.60, 0.78]
    },
    'IPA Transliterate': {
        'sizes': [0.4e9, 1.3e9, 7.1e9, 13e9, 70e9, 175e9, 540e9],
        'accuracy': [0.0, 0.0, 0.0, 0.02, 0.08, 0.42, 0.70]
    }
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

for idx, (task_name, data) in enumerate(tasks.items()):
    axes[idx].semilogx(data['sizes'], data['accuracy'], 'o-',
                       color=colors[idx], linewidth=2, markersize=8)
    axes[idx].set_xlabel('Model Parameters')
    axes[idx].set_ylabel('Accuracy (exact match)')
    axes[idx].set_title(task_name)
    axes[idx].set_ylim(-0.05, 1.0)
    axes[idx].grid(True, alpha=0.3)
    # Shade the "emergence zone"
    axes[idx].axvspan(10e9, 200e9, alpha=0.1, color=colors[idx])

plt.suptitle('Emergent Abilities in LLMs (approx. data from Wei et al., 2022)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Simulate grokking: a real phase transition in learning
# Train a small model on modular arithmetic and show the sudden generalization

def generate_modular_data(p: int = 97, op: str = 'add'):
    """Generate all (a op b) mod p examples."""
    data = []
    for a in range(p):
        for b in range(p):
            if op == 'add':
                c = (a + b) % p
            elif op == 'mult':
                c = (a * b) % p
            data.append((a, b, c))
    return data

p = 47
all_data = generate_modular_data(p, 'add')
np.random.shuffle(all_data)
split = int(0.3 * len(all_data))  # 30% train, 70% test (intentionally small train set)
train_data = all_data[:split]
test_data = all_data[split:]

# Simple embedding + MLP model
class GrokModel(nn.Module):
    def __init__(self, p, d_model=64):
        super().__init__()
        self.emb_a = nn.Embedding(p, d_model)
        self.emb_b = nn.Embedding(p, d_model)
        self.mlp = nn.Sequential(
            nn.Linear(2 * d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, p)
        )

    def forward(self, a, b):
        x = torch.cat([self.emb_a(a), self.emb_b(b)], dim=-1)
        return self.mlp(x)

grok_model = GrokModel(p).to(device)
optimizer = torch.optim.AdamW(grok_model.parameters(), lr=1e-3, weight_decay=1.0)

train_a = torch.tensor([d[0] for d in train_data], device=device)
train_b = torch.tensor([d[1] for d in train_data], device=device)
train_c = torch.tensor([d[2] for d in train_data], device=device)
test_a = torch.tensor([d[0] for d in test_data], device=device)
test_b = torch.tensor([d[1] for d in test_data], device=device)
test_c = torch.tensor([d[2] for d in test_data], device=device)

train_losses, test_losses, train_accs, test_accs = [], [], [], []

for epoch in range(3000):
    # Train
    grok_model.train()
    logits = grok_model(train_a, train_b)
    loss = F.cross_entropy(logits, train_c)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % 20 == 0:
        grok_model.eval()
        with torch.no_grad():
            train_pred = grok_model(train_a, train_b).argmax(-1)
            test_pred = grok_model(test_a, test_b).argmax(-1)
            train_acc = (train_pred == train_c).float().mean().item()
            test_acc = (test_pred == test_c).float().mean().item()
            test_loss = F.cross_entropy(grok_model(test_a, test_b), test_c).item()

        train_losses.append(loss.item())
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)

print(f"Final train acc: {train_accs[-1]:.3f}, test acc: {test_accs[-1]:.3f}")

In [ ]:
# Plot grokking phase transition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = np.arange(len(train_accs)) * 20

axes[0].plot(epochs, train_accs, label='Train', color='steelblue', linewidth=2)
axes[0].plot(epochs, test_accs, label='Test', color='darkorange', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Grokking: Sudden Generalization Phase Transition')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(epochs, train_losses, label='Train', color='steelblue', linewidth=2)
axes[1].semilogy(epochs, test_losses, label='Test', color='darkorange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Grokking: Loss Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\nGrokking demonstrates a genuine phase transition: the model memorizes training data")
print("quickly but generalizes to unseen data only after much longer training with weight decay.")

## Summary

This notebook covered the expert-level topics in pretraining:

1. **Scaling laws** — Kaplan (bigger models, less data) vs Chinchilla (equal scaling). Chinchilla changed the industry.

2. **Compute-optimal training** — C = 6ND, and the optimal ratio is D ~ 20N. But inference cost pushes toward overtraining smaller models.

3. **Data mixtures** — Weighted domain sampling outperforms proportional sampling. High-quality small domains (books, code) are upsampled.

4. **Data quality** — MinHash deduplication removes near-duplicates at scale. Perplexity filtering removes garbage and boilerplate.

5. **Training dynamics** — Monitor loss, gradient norms, and update ratios. Detect spikes early. Layer-wise stats catch hidden instabilities.

6. **Emergent abilities** — The metric-choice argument explains many apparent emergences, but genuine phase transitions (like grokking) do exist.

**Next:** [05-pretraining/build.ipynb](./build.ipynb) — pretrain a small language model from scratch.